In [24]:
example = [
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
]

In [6]:
import pandas as pd
from tqdm import tqdm

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
# lets generate the index for search
from ingest import build_index, load_faq_data

documents = load_faq_data()
documents_llm_zoomcamp = [] 
for d in documents:
    if d["course"] == "llm-zoomcamp":
        documents_llm_zoomcamp.append(d)

documents = documents_llm_zoomcamp
index = build_index(documents)

# search function using text
def text_search(query):
    boost_dict = {
        "question": 3.0,
        "section": 0.5
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )


In [7]:
# function that returns a relevante list for each query
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [8]:
relevance_total = compute_relevance_total(ground_truth, text_search)

100%|██████████| 805/805 [00:02<00:00, 284.25it/s]


In [11]:
relevance_total[:5]

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0]]

In [10]:
# HIT RATE

example = relevance_total[:5]

cnt = 0

for line in example:
    if 1 in line:
        cnt = cnt + 1

cnt

2

In [13]:
print(cnt / len(example))

0.4


In [14]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [ ]:
#hit_rate(relevance_total) # 0.8347826086956521
hit_rate(example) # 0.9333333333333333

0.9333333333333333

In [28]:
# MRR Mean Reciprocal Rank

# Logic for MRR
l = [0,0,1,0,0] # 33%
l = [0,0,0,1,0] # 25%
l = [0,0,0,0,1] # 20%
l = [1,0,0,0,0] # 100%
l = [0,1,0,0,0] # 50%
l=example[1]

print(l)

total_count = 0.0
for x in range(len(l)):
    if l[x] == 1:
        weight = 1 / (x + 1)
        break
print(weight)

[0, 1, 0, 0, 0]
0.5


In [30]:
total_score = 0.0

for line in example:
    for rank in range(len(line)):
        if line[rank] == 1:
            total_score = total_score + 1 / (rank + 1)
            break

total_score
total_score / len(example)

0.8222222222222222

In [ ]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)

In [32]:
mrr(example)
# 0.822

0.8222222222222222

In [33]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [34]:
evaluate(
    ground_truth,
    text_search
)

100%|██████████| 805/805 [00:03<00:00, 254.73it/s]


{'hit_rate': 0.8347826086956521, 'mrr': 0.7110144927536229}